# Optimal Synaptic Profiles

Generates Figs. 2, 3, S9, S10, and S11. This notebook uses the weak-coupling gradients for a traveling-wave input on a ring to compute Fisher-information-optimizing profiles of release probability $U(\Delta z)$, postsynaptic weight $w_0(\Delta z)$, and effective connectivity under the resource constraints described in the manuscript.


In [ ]:
# Imports
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
from scipy import signal
from numba import jit

from pathlib import Path

ROOT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
FIGURE_DIR = ROOT_DIR / "figures"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
# Shared plotting style for manuscript PDFs.
config = {
    "font.family": "sans-serif",
    "font.size": 18.0,
    "axes.titlelocation": "left",
    "axes.titlesize": 19.0,
    "axes.labelsize": 19.0,
    "xtick.labelsize": 17.0,
    "ytick.labelsize": 17.0,
    "legend.fontsize": 17.0,
    "figure.titlesize": 19.0,
    "axes.linewidth": 0.8,
    "lines.linewidth": 1.5,
    "lines.markersize": 4.0,
    "patch.linewidth": 0.8,
    "xtick.direction": "in",
    "ytick.direction": "in",
    "axes.xmargin": 0.01,
    "axes.ymargin": 0.05,
    "xtick.major.size": 3.5,
    "ytick.major.size": 3.5,
    "xtick.major.width": 0.8,
    "ytick.major.width": 0.8,
    "xtick.minor.size": 2.0,
    "ytick.minor.size": 2.0,
    "xtick.minor.width": 0.6,
    "ytick.minor.width": 0.6,
    "legend.frameon": False,
    "legend.fancybox": False,
    "image.interpolation": "none",
    "savefig.dpi": 300,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "svg.fonttype": "none",
}
plt.rcParams.update(config)


In [ ]:
# Define activation functions
def activation_function_exp(u, params):
    """Exponential firing-rate nonlinearity."""
    return params['g_M'] * np.exp(params['beta'] * (u - params['u_c']))

def activation_function_sigmoid(u, params):
    """Sigmoid firing-rate nonlinearity."""
    from scipy.special import expit
    return params['g_M'] * expit(params['beta'] * (u - params['u_c']))

@jit(nopython=True)
def convolve_epsp(signal_in, tau_s, dt):
    """ Convolve input signal with EPSP kernel using recursive filter. """
    a = np.exp(-dt/tau_s)
    b = tau_s * (1 - a)
    signal_out = np.zeros_like(signal_in)
    for i in range(1, len(signal_in)):
        signal_out[i] = a * signal_out[i-1] + b * signal_in[i-1]
    return signal_out


In [ ]:
# compute pre/post factors
# compute pre factor
def compute_pre_factor(h_pre, t_eval, U, w0, params, activation='exp'):
    """
    Compute the pre-synaptic factor for gradient calculation with STP.
        h_pre: pre-synaptic (external) input (func)
        t_eval: time points for evaluation (array)
        U: release prob.
        w0: post-synaptic weight
        activation: type of activation function ('exp' or 'sigmoid')
    """
    if activation == 'exp':
        activation_function = lambda u: activation_function_exp(u, params=params)
    elif activation == 'sigmoid':
        activation_function = lambda u: activation_function_sigmoid(u, params=params)
    else:
        raise ValueError("Unsupported activation function")
    def dynamics_STP_factor(t, y, U, nu0_func, tau_d):
        '''
        dynamics of STP factor.
            w: synaptic efficacy
            f_w0: derivative with respect to w0
            f_U: derivative with respect to U
            U: release prob.
            nu0_func: pre firing rate (as func of time)
            tau_d: STD time constant
        '''
        f_w0, f_U = y
        nu0_t = nu0_func(t)

        recovery = 1 / tau_d
        usage = nu0_t * U

        df_w0_dt = recovery * (1-f_w0) - usage * f_w0
        df_U_dt = recovery * (1 - f_U) - usage * (f_U + f_w0)
        return [df_w0_dt, df_U_dt]

    # unpack parameters
    tau_d = params['tau_d']
    tau_s = params['tau_s'] # EPSP time constant (s)
    t = t_eval
    t0 = t[0]
    t1 = t[-1]
    dt = t[1] - t[0]

    nu0_pre = activation_function(h_pre(t)) # array (Nt,) : pre firing rate

    # compute STP factor
    nu0_func = lambda t_val: activation_function(h_pre(t_val))
    f0 = [1/(1 + tau_d * nu0_pre[0] * U),1/(1 + tau_d * nu0_pre[0] * U)**2] # init values for f = [f_w0, f_U]


    # define max_step
    if 'freq' in params and params['freq'] and params['freq'] > 0:
        max_step = 1 / (10 * params['freq'])
    else:
        max_step = dt * 10

    sol_f = solve_ivp(fun=lambda t, y: dynamics_STP_factor(t, y, U, nu0_func, tau_d),
        t_span = (t0, t1),
        y0=f0,
        t_eval=t,
        method='BDF',
        max_step=max_step
    )

    f_w0 = sol_f.y[0,:] # array (Nt,) : f_w0 at each time point
    f_U = sol_f.y[1,:] # array (Nt,) : f_U at each time point
    pre_factor_w0 = nu0_pre * f_w0 * U
    pre_factor_U = nu0_pre * f_U * w0
    pre_factor_w_noSTP = nu0_pre

    return {
        'pre_factor_w0': pre_factor_w0,
        'pre_factor_U': pre_factor_U,
        'pre_factor_w_noSTP': pre_factor_w_noSTP,
        'nu0_pre': nu0_pre,
        'h_pre': h_pre(t),
        'f_w0': f_w0,
        'f_U': f_U
    }
# compute post factor
def compute_post_factor(h_post, h_prime, t_eval, params, activation='exp'):
    """
    Compute the post-synaptic factor for gradient calculation.
        h_post: post-synaptic (external) input (func)
        h_prime: derivative of h_post (with respect to encoding param) (func)
        t_eval: time points for evaluation (array)
        activation: type of activation function ('exp' or 'sigmoid')
    """

    if activation == 'exp':
        activation_function = lambda u: activation_function_exp(u, params=params)
    elif activation == 'sigmoid':
        activation_function = lambda u: activation_function_sigmoid(u, params=params)
    else:
        raise ValueError("Unsupported activation function")

    t = t_eval
    t0 = t[0]
    t1 = t[-1]
    dt = t[1] - t[0]

    nu0_post = activation_function(h_post(t)) # array (Nt,) : post firing rate

    if activation == 'exp':
        eta = (params['beta'] ** 3) * (h_prime(t) ** 2)
    elif activation == 'sigmoid':
        eta = (params['beta'] ** 3) * (h_prime(t) ** 2) * ((1 - nu0_post / params['g_M']) ** 2) * (1 - 3 * nu0_post / params['g_M'])
    post_factor = nu0_post * eta
    return {'post_factor': post_factor, 'nu0_post': nu0_post, 'h_post': h_post(t), 'h_prime': h_prime(t), 'eta': eta}

# compute gradient
def compute_gradient(pre_factor, post_factor, t_eval, params, t_span_grad=None):
    """
    Compute the gradient of the synaptic weight with respect to encoding parameters.
        pre_factor: pre-synaptic factor (array)
        post_factor: post-synaptic factor (array)
        t_eval: time points for evaluation (array)
        tau_s: EPSP time constant (s)
    """
    tau_s = params['tau_s']
    dt = t_eval[1] - t_eval[0]

    # Convolve pre_factor with EPSP kernel
    convolved_pre = convolve_epsp(pre_factor, tau_s, dt)
    pre_post_product = convolved_pre * post_factor

    if t_span_grad is not None:
        # Restrict to specified time span for gradient calculation
        mask = (t_eval >= t_span_grad[0]) & (t_eval <= t_span_grad[1])
    else:
        mask = np.ones_like(t_eval, dtype=bool)
    # Compute gradient as integral of convolved pre and post factors
    gradient = np.trapezoid(pre_post_product[mask], t_eval[mask])

    return { 'gradient': gradient, 'pre_post_product': pre_post_product }
def compute_gradient_all(result_pre, result_post, t_eval, params, t_span_grad=None):
    """ Compute gradients for all pre factors. """
    gradients = {}
    pre_post_products = {}
    for key in ['pre_factor_w0', 'pre_factor_U', 'pre_factor_w_noSTP']:
        res = compute_gradient(result_pre[key], result_post['post_factor'], t_eval, params, t_span_grad)
        output_key = key.replace('pre_factor_', '')
        gradients[output_key] = res['gradient']
        pre_post_products[output_key] = res['pre_post_product']
    return gradients, pre_post_products


In [ ]:
# plot pre / post factors


In [ ]:
def shift_pre_factor_by_phase(result_pre_0, delta_z, t_eval, params):
    """

    Parameters:
    -----------
    result_pre_0 : dict
    delta_z : float
    t_eval : array
    params : dict

    Returns:
    --------
    result_pre_delta_z : dict
    """
    pre_phase = -delta_z  # Δz = z_post - z_pre, z_post is fixed at 0
    pre_phase = (pre_phase + 2 * np.pi) % (2 * np.pi)
    dt = t_eval[1] - t_eval[0]
    freq = params['freq']

    time_shift = -pre_phase / (2 * np.pi * freq)

    index_shift = int(np.round(time_shift / dt))

    result_pre_delta_z = {}
    for key in result_pre_0.keys():
        if isinstance(result_pre_0[key], np.ndarray):
            result_pre_delta_z[key] = np.roll(result_pre_0[key], -index_shift)
        else:
            result_pre_delta_z[key] = result_pre_0[key]

    return result_pre_delta_z


In [ ]:
def compute_grad_U_matrix(h, h_prime, delta_z_arr, U_arr, params,  w0_init=1.0):
    """
    Compute gradient matrix for U.
    Parameters:
    -----------
        h: external input function (func of pre-synaptic phase and t)
        h_prime: derivative of h with respect to encoding param (func of pre-synaptic phase, t)
        delta_z_arr: array of phase offsets Δz (= z_post - z_pre)
        U_arr: array of release probabilities
        w0_init: initial post-synaptic weight (scalar)
    Returns:
    --------
        grad_U_matrix: 2D array of gradients with shape (len(U_arr), len(delta_z_arr))
    """

    h_post = lambda t: h(0, t)
    h_prime_ = lambda t: h_prime(0, t)

    h_pre_0 = lambda t: h(0, t) # at Δz = 0

    t0 = -0.5/params['freq']; t1 = 10.0 / params['freq']
    dt = 0.001
    t_eval = np.arange(t0, t1, dt)
    t_span_grad = (t1 - 3.5 / params['freq'], t1 - 1.5 / params['freq'])

    grad_U_matrix = np.empty((len(U_arr), len(delta_z_arr)))

    result_post = compute_post_factor(h_post, h_prime_, t_eval, params=params, activation='exp')

    for i in range(len(U_arr)):
        U = U_arr[i]
        result_pre_0 = compute_pre_factor(h_pre_0, t_eval, U=U, w0=w0_init, params=params, activation='exp')
        for j, delta_z in enumerate(delta_z_arr):
            result_pre_delta_z = shift_pre_factor_by_phase(result_pre_0, delta_z, t_eval, params)
            gradients, _ = compute_gradient_all(result_pre_delta_z, result_post, t_eval, params, t_span_grad)
            grad_U_matrix[i, j] = gradients['U']
    return grad_U_matrix

def plot_grad_U_matrix(grad_U_matrix, delta_z_arr, U_arr):
    """ Plot gradient matrix for U. """
    fig, ax = plt.subplots(figsize=(8, 6))
    vmin, vmax = np.percentile(grad_U_matrix, [1, 99])
    im = ax.imshow(grad_U_matrix, extent=[delta_z_arr[0], delta_z_arr[-1], U_arr[0], U_arr[-1]], aspect='auto', origin='lower', cmap='viridis', vmin=vmin, vmax=vmax)
    cbar = fig.colorbar(im, ax=ax)
    cbar.set_label(r'$\delta J/\delta U$')
    levels = np.geomspace(1e-2, vmax, 20)
    cs = ax.contour(delta_z_arr, U_arr, grad_U_matrix, levels=levels[10:], colors='white', linewidths=0.5, alpha=0.7)
    cs_0 = ax.contour(delta_z_arr, U_arr, grad_U_matrix, levels=[0], colors='red', linewidths=2)
    ax.set_xlabel('Phase offset Δz (rad)')
    ax.set_ylabel('Release probability U')
    ax.set_title('Gradient Matrix for U')
    plt.show()
    return fig, ax


In [ ]:
### (technical) functions for computing U(Δz) from grad_U(Δz, U)

def find_U_for_gradient_level(grad_U_matrix, delta_z_arr, U_arr, grad_level):
    """
    """
    U_z = np.zeros_like(delta_z_arr)

    for i, delta_z in enumerate(delta_z_arr):
        grad_profile = grad_U_matrix[:, i]

        if grad_level > np.max(grad_profile):
            U_z[i] = U_arr[0]
        elif grad_level < np.min(grad_profile):
            U_z[i] = U_arr[-1]
            continue
        idx = np.argmax(grad_level >= grad_profile)
        if idx == 0:
            U_z[i] = U_arr[0]
        else:
            g1, g2 = grad_profile[idx-1], grad_profile[idx]
            U1, U2 = U_arr[idx-1], U_arr[idx]
            if g2 - g1 != 0:
                alpha = (grad_level - g1) / (g2 - g1)
                U_z[i] = U1 + alpha * (U2 - U1)
            else:
                U_z[i] = U1
    return U_z

def calculate_positive_fraction(U_z):
    """"""
    return np.mean(U_z > 1.0 / len(U_z))

def find_gradient_level_for_sparsity_constraint(grad_U_matrix, delta_z_arr, U_arr, target_fraction=0.5, max_iter=100):
    grad_min = grad_U_matrix.min() + 1e-2
    grad_max = grad_U_matrix.max() - 1e-2

    test_levels = np.linspace(grad_min, grad_max, 50)
    fractions = np.array([calculate_positive_fraction(find_U_for_gradient_level(grad_U_matrix, delta_z_arr, U_arr, level))
                          for level in test_levels])

    closest_idx = np.argmin(np.abs(fractions - target_fraction))

    if closest_idx > 0:
        left = test_levels[closest_idx - 1]
    else:
        left = grad_min

    if closest_idx < len(test_levels) - 1:
        right = test_levels[closest_idx + 1]
    else:
        right = grad_max

    for _ in range(max_iter):
        mid = (left + right) / 2
        U_mid = find_U_for_gradient_level(grad_U_matrix, delta_z_arr, U_arr, mid)
        fraction = calculate_positive_fraction(U_mid)

        if abs(fraction - target_fraction) < 1.0 / len(delta_z_arr):
            grad_level = mid
            U_z = U_mid
            info = {'converged': True, 'iterations': _+1, 'final_fraction': fraction}
            return grad_level, U_z, info

        if fraction > target_fraction:
            left = mid
        else:
            right = mid
    grad_level = (left + right) / 2
    U_z = find_U_for_gradient_level(grad_U_matrix, delta_z_arr, U_arr, grad_level)
    info = {'converged': False, 'iterations': max_iter, 'final_fraction': calculate_positive_fraction(U_z)}
    return grad_level, U_z, info

def find_gradient_level_for_mean_constraint(grad_U_matrix, delta_z_arr, U_arr, target_mean, tol=1e-2, max_iter=100):
    """

    Parameters:
    -----------
    grad_U_matrix : ndarray
    delta_z_arr : ndarray
    U_arr : ndarray
    target_mean : float
    tol : float
    max_iter : int

    Returns:
    --------
    grad_level : float
    U_z : ndarray
    info : dict
    """

    grad_min = grad_U_matrix.min()
    grad_max = grad_U_matrix.max()

    def compute_mean(grad_level):
        U_z = find_U_for_gradient_level(grad_U_matrix, delta_z_arr, U_arr, grad_level)
        return np.mean(U_z)

    mean_at_min = compute_mean(grad_min)
    mean_at_max = compute_mean(grad_max)

    is_increasing = mean_at_max > mean_at_min

    if not (min(mean_at_min, mean_at_max) <= target_mean <= max(mean_at_min, mean_at_max)):
        print(f"Warning: target_mean {target_mean:.4f} is outside achievable range [{min(mean_at_min, mean_at_max):.4f}, {max(mean_at_min, mean_at_max):.4f}]")
        if target_mean < min(mean_at_min, mean_at_max):
            grad_level = grad_max if mean_at_max < mean_at_min else grad_min
        else:
            grad_level = grad_min if mean_at_min > mean_at_max else grad_max
        U_z = find_U_for_gradient_level(grad_U_matrix, delta_z_arr, U_arr, grad_level)
        return grad_level, U_z, {'converged': False, 'mean': compute_mean(grad_level), 'iterations': 0}

    left, right = grad_min, grad_max

    for iteration in range(max_iter):
        grad_level = (left + right) / 2
        current_mean = compute_mean(grad_level)

        if abs(current_mean - target_mean) < tol:
            U_z = find_U_for_gradient_level(grad_U_matrix, delta_z_arr, U_arr, grad_level)
            return grad_level, U_z, {'converged': True, 'mean': current_mean, 'iterations': iteration + 1}

        if is_increasing:
            if current_mean < target_mean:
                left = grad_level
            else:
                right = grad_level
        else:
            if current_mean < target_mean:
                right = grad_level
            else:
                left = grad_level

    U_z = find_U_for_gradient_level(grad_U_matrix, delta_z_arr, U_arr, grad_level)
    print(f"Warning: Maximum iterations ({max_iter}) reached. Current error: {abs(current_mean - target_mean):.6f}")
    return grad_level, U_z, {'converged': False, 'mean': current_mean, 'iterations': max_iter}

def find_gradient_level_for_maximum_constraint(grad_U_matrix, delta_z_arr, U_arr, target_max = 0.7, tol=1e-2, max_iter=100):
    """

    Parameters:
    -----------
    grad_U_matrix : ndarray
    delta_z_arr : ndarray
    U_arr : ndarray
    target_max : float
    tol : float
    max_iter : int

    Returns:
    --------
    grad_level : float
    U_z : ndarray
    info : dict
    """

    grad_min = grad_U_matrix.min()
    grad_max = grad_U_matrix.max()

    def compute_max(grad_level):
        U_z = find_U_for_gradient_level(grad_U_matrix, delta_z_arr, U_arr, grad_level)
        return np.max(U_z)

    max_at_min = compute_max(grad_min)
    max_at_max = compute_max(grad_max)

    is_increasing = max_at_max > max_at_min

    if not (min(max_at_min, max_at_max) <= target_max <= max(max_at_min, max_at_max)):
        print(f"Warning: target_max {target_max:.4f} is outside achievable range [{min(max_at_min, max_at_max):.4f}, {max(max_at_min, max_at_max):.4f}]")
        if target_max < min(max_at_min, max_at_max):
            grad_level = grad_max if max_at_max < max_at_min else grad_min
        else:
            grad_level = grad_min if max_at_min > max_at_max else grad_max
        U_z = find_U_for_gradient_level(grad_U_matrix, delta_z_arr, U_arr, grad_level)
        return grad_level, U_z, {'converged': False, 'mean': compute_max(grad_level), 'iterations': 0}

    left, right = grad_min, grad_max

    for iteration in range(max_iter):
        grad_level = (left + right) / 2
        current_max = compute_max(grad_level)

        if abs(current_max - target_max) < tol:
            U_z = find_U_for_gradient_level(grad_U_matrix, delta_z_arr, U_arr, grad_level)
            return grad_level, U_z, {'converged': True, 'mean': current_max, 'iterations': iteration + 1}

        if is_increasing:
            if current_max < target_max:
                left = grad_level
            else:
                right = grad_level
        else:
            if current_max < target_max:
                right = grad_level
            else:
                left = grad_level

    U_z = find_U_for_gradient_level(grad_U_matrix, delta_z_arr, U_arr, grad_level)
    print(f"Warning: Maximum iterations ({max_iter}) reached. Current error: {abs(current_max - target_max):.6f}")
    return grad_level, U_z, {'converged': False, 'mean': current_max, 'iterations': max_iter}


In [ ]:
def compute_weight_from_params(h, h_prime, params):
    if 'activation' in params:
        activation = params['activation']
    else:
        activation = 'exp'
    h_post = lambda t: h(0, t)
    h_prime_ = lambda t: h_prime(0, t)

    delta_z_arr = np.linspace(-np.pi, np.pi, 100)
    U_init = params['U_init']
    U_arr = np.linspace(1e-3, 1.0-1e-3, 50)
    w0_init = params['w0_init']

    # compute grad_U_matrix
    grad_U_matrix = compute_grad_U_matrix(h, h_prime, delta_z_arr, U_arr, params, w0_init=w0_init)

    # compute optimal U(Δz)
    # check constraints
    assert (('target_fraction' in params) + ('target_mean' in params) + ('target_max' in params)) <= 1, "Only one constraint can be specified among target_fraction, target_mean, target_max."
    if 'target_fraction' in params:
        target_fraction = params['target_fraction']
        grad_level, U_z, info = find_gradient_level_for_sparsity_constraint(grad_U_matrix, delta_z_arr, U_arr, target_fraction=target_fraction)
    elif 'target_max' in params:
        target_max = params['target_max']
        grad_level, U_z, info = find_gradient_level_for_maximum_constraint(grad_U_matrix, delta_z_arr, U_arr, target_max=target_max)
    elif 'target_mean' in params:
        target_mean = params['target_mean']
        grad_level, U_z, info = find_gradient_level_for_mean_constraint(grad_U_matrix, delta_z_arr, U_arr, target_mean=target_mean)
    else:
        grad_level, U_z, info = find_gradient_level_for_mean_constraint(grad_U_matrix, delta_z_arr, U_arr, target_mean=U_init)
    print(f"Gradient level found: {grad_level:.4f}, info: {info}")
    U_opt = U_z

    t0 = -0.5/params['freq']; t1 = 10.0 / params['freq']
    dt = 0.001
    t_eval = np.arange(t0, t1, dt)
    t_span_grad = (t1 - 3.5 / params['freq'], t1 - 1.5 / params['freq'])
    t_span_plot = (t1 - 3.5 / params['freq'], t1 - 1.5 / params['freq'])

    result_post = compute_post_factor(h_post, h_prime_, t_eval, params=params, activation=activation)

    grad_w0_optU_arr = np.empty_like(delta_z_arr)
    grad_w0_initU_arr = np.empty_like(delta_z_arr)

    for i, delta_z in enumerate(delta_z_arr):
        pre_phase = -delta_z  # Δz = z_post - z_pre, z_post is fixed at 0
        h_pre = lambda t: h(pre_phase, t)
        result_pre_Uopt = compute_pre_factor(h_pre, t_eval, U=U_opt[i], w0=w0_init, params=params, activation=activation)
        result_pre_initU = compute_pre_factor(h_pre, t_eval, U=U_init, w0=w0_init, params=params, activation=activation)
        gradients, _ = compute_gradient_all(result_pre_Uopt, result_post, t_eval, params, t_span_grad)
        grad_w0_optU_arr[i] = gradients['w0']

        gradients, _ = compute_gradient_all(result_pre_initU, result_post, t_eval, params, t_span_grad)
        grad_w0_initU_arr[i] = gradients['w0']

    def normalize_by_mean_std(arr):
        arr_ = arr - np.mean(arr)
        arr_ = arr_ / np.std(arr_)
        return arr_
    w0_opt_optU = normalize_by_mean_std(grad_w0_optU_arr)
    w0_opt_initU = normalize_by_mean_std(grad_w0_initU_arr)

    return {
        'delta_z_arr': delta_z_arr,
        'U_arr': U_arr,
        'U_opt': U_opt,
        'w0_opt_optU': w0_opt_optU,
        'w0_opt_initU': w0_opt_initU,
        'grad_U_matrix': grad_U_matrix
    }


In [ ]:
# Baseline parameter set for the published figures.

params = {
    'freq': 1.0,
    'amp': 2.0,
    'tau_d': 0.5,  # Depression time constant (s)
    'beta' : 2.0,  # Steepness of activation function
    'g_M' : 10.0, # Maximum firing rate (Hz) for sigmoid activation.
    'u_c' : 1.0,    # Activation threshold
    'tau_s' : 0.01,  # Synaptic time constant (s),
    'U_init' : 0.15,
    'w0_init' : 1.0,
    'activation' : 'exp',
}

rectified_cos = lambda theta , theta_c: np.maximum(0, (np.cos(theta) - np.cos(theta_c)))
# define h functions
h_rect = lambda z_pre, t: params['amp'] * rectified_cos(2 * np.pi * params['freq'] * t - z_pre, theta_c=np.pi/2)
h_prime = lambda z_pre, t: np.where(h_rect(z_pre, t) > 0, -params['amp'], 0)

result_weight = compute_weight_from_params(h_rect, h_prime, params)


In [ ]:
# Plot the gradient matrix.
grad_U_matrix = result_weight['grad_U_matrix']
delta_z_arr = result_weight['delta_z_arr']
U_arr = result_weight['U_arr']
grad_level, U_z, info = find_gradient_level_for_mean_constraint(grad_U_matrix, delta_z_arr, U_arr, target_mean=params['U_init'])

fig, ax = plt.subplots(1, 1, figsize=(12, 8))
vmin, vmax = np.percentile(grad_U_matrix, [2, 97])
im = ax.imshow(
    np.maximum(grad_U_matrix, 1e-6),
    extent=[delta_z_arr[0], delta_z_arr[-1], U_arr[0], U_arr[-1]],
    aspect='auto',
    origin='lower',
    cmap='viridis',
    vmin=vmin,
    vmax=vmax,
)
cbar = fig.colorbar(im, ax=ax)
cbar.set_label(r'$\delta J/\delta U$')
# ax.imshow(grad_U_matrix, extent=[delta_z_arr[0], delta_z_arr[-1], U_arr[0], U_arr[-1]], aspect='auto', origin='lower', cmap='viridis')
# cbar = fig.colorbar(ax.images[0], ax=ax)
# cbar.set_label(r'$\delta J/\delta U$')
levels = np.linspace(np.maximum(1e-2, vmin), vmax, 10)
cs = ax.contour(delta_z_arr, U_arr, grad_U_matrix, levels=levels[:], colors='white', linewidths=0.7)
cs_0 = ax.contour(delta_z_arr, U_arr, grad_U_matrix, levels=[0], colors='red', linewidths=2)
ax.set_xlabel(r'Phase offset Δz (rad)')
ax.set_ylabel(r'Release probability $U$')

plt.tight_layout()
plt.savefig(FIGURE_DIR / 'suppl-grad-U.pdf', bbox_inches='tight')
plt.show()


In [ ]:
fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(18, 5), sharex=True)

ax1.plot(result_weight['delta_z_arr'], result_weight['U_opt'], color='blue')
ax1.plot(result_weight['delta_z_arr'], params['U_init']*np.ones_like(result_weight['delta_z_arr']), color='blue', linestyle='--')
ax1.axvline(0, color='black', linestyle=':', linewidth=0.8)
ax1.axhline(0, color='black', linestyle=':', linewidth=0.8)
ax1.set_ylabel(r'Release probability $U$')
ax1.set_xlabel(r'Anti-causal $\leftarrow$ $\Delta z$ (rad) $\rightarrow$ Causal')
# ax1.set_title('Optimal Release Probability U(Δz)')
ax1.set_title('A')
print('A: Optimal Release Probability  U')
ax1.grid(True, alpha=0.3)
# ax1.legend(loc='upper left', bbox_to_anchor=(1.02, 1), borderaxespad=0)

ax2.plot(result_weight['delta_z_arr'], result_weight['w0_opt_optU'], color='green')
ax2.plot(result_weight['delta_z_arr'], result_weight['w0_opt_initU'], color='green', linestyle='--')
ax2.axhline(0, color='black', linestyle=':', linewidth=0.8)
ax2.axvline(0, color='black', linestyle=':', linewidth=0.8)
ax2.set_ylabel(r'Postsynaptic weight $w_0$')
ax2.set_xlabel(r'Anti-causal $\leftarrow$ $\Delta z$ (rad) $\rightarrow$ Causal')
ax2.set_title('B')
print('B: Optimal Postsynaptic Weight ')
ax2.grid(True, alpha=0.3)
# ax2.legend(loc='upper left', bbox_to_anchor=(1.02, 1), borderaxespad=0)

ax3.plot(result_weight['delta_z_arr'], result_weight['w0_opt_optU']*result_weight['U_opt'],color='purple', label = 'U: plastic')
ax3.plot(result_weight['delta_z_arr'], result_weight['w0_opt_initU']*params['U_init'], color='purple', linestyle='--', label = 'U: fixed')
ax3.axhline(0, color='black', linestyle=':', linewidth=0.8)
ax3.axvline(0, color='black', linestyle=':', linewidth=0.8)
ax3.set_ylabel(r'Effective connectivity $w_{\rm eff}$')
ax3.set_xlabel(r'Anti-causal $\leftarrow$ $\Delta z$ (rad) $\rightarrow$ Causal')
ax3.set_title('C')
print('C: Effective Synaptic Weight w_eff(Δz) = w0(Δz)*U(Δz)')
ax3.grid(True, alpha=0.3)
ax3.legend(loc='upper left', bbox_to_anchor=(1.02, 1), borderaxespad=0)
plt.tight_layout()
plt.savefig(FIGURE_DIR / 'optimal-profiles.pdf', bbox_inches='tight')
plt.show()


In [ ]:
# comparison between awake and sleep
import copy
params_awake = copy.deepcopy(params)
params_sleep = copy.deepcopy(params)
params_sleep['U_init'] = 0.03  # lower release probability in sleep

result_weight_awake = copy.deepcopy(result_weight)
result_weight_sleep = compute_weight_from_params(h_rect, h_prime, params_sleep)

fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(18, 5), sharex=True)
ax1.plot(result_weight_awake['delta_z_arr'], result_weight_awake['U_opt'], label='U: high', color='blue')
ax1.plot(result_weight_sleep['delta_z_arr'], result_weight_sleep['U_opt'], label='U: low', color='blue', linestyle='--')
ax1.axhline(0, color='black', linestyle=':', linewidth=0.8)
ax1.axvline(0, color='black', linestyle=':', linewidth=0.8)
ax1.set_ylabel(r'Release probability $U$')
ax1.set_xlabel(r'Anti-causal $\leftarrow$ $\Delta z$ (rad) $\rightarrow$ Causal')
print('A: Optimal Release Probability  U')
ax1.set_title('A')
ax1.grid(True, alpha=0.3)

ax2.plot(result_weight_awake['delta_z_arr'], result_weight_awake['w0_opt_optU'], label='U: high', color='green')
ax2.plot(result_weight_sleep['delta_z_arr'], result_weight_sleep['w0_opt_optU'], label='U: low', color='green', linestyle='--')
ax2.axhline(0, color='black', linestyle=':', linewidth=0.8)
ax2.axvline(0, color='black', linestyle=':', linewidth=0.8)
ax2.set_ylabel(r'Postsynaptic weight $w_0$')
ax2.set_xlabel(r'Anti-causal $\leftarrow$ $\Delta z$ (rad) $\rightarrow$ Causal')
ax2.set_title('B')
print('B: Optimal Postsynaptic Weight ')
ax2.grid(True, alpha=0.3)
# ax2.legend(loc='upper left', bbox_to_anchor=(1.02, 1), borderaxespad=0)

ax3.plot(result_weight_awake['delta_z_arr'], result_weight_awake['w0_opt_optU']*result_weight_awake['U_opt'], label='U: high', color='purple')
ax3.plot(result_weight_sleep['delta_z_arr'], result_weight_sleep['w0_opt_optU']*result_weight_sleep['U_opt'], label='U: low', color='purple', linestyle='--')
ax3.set_ylabel(r'Effective connectivity $w_{\rm eff}$')
ax3.set_xlabel(r'Anti-causal $\leftarrow$ $\Delta z$ (rad) $\rightarrow$ Causal')
print('C: Effective Synaptic Weight w_eff(Δz)')
ax3.axhline(0, color='black', linestyle=':', linewidth=0.8)
ax3.axvline(0, color='black', linestyle=':', linewidth=0.8)
ax3.set_title('C')
ax3.grid(True, alpha=0.3)
ax3.legend(loc='upper left', bbox_to_anchor=(1.02, 1), borderaxespad=0)
plt.tight_layout()
plt.savefig(FIGURE_DIR / 'optimal-profiles-awake-sleep.pdf', bbox_inches='tight')
plt.show()


### Supplementary Figure: Optimal $U(\Delta z)$ under different constraints

Reproduces Fig. S11 by varying the release-probability constraints while keeping the traveling-wave setting fixed.


In [ ]:
# Compare optimized U under several constraints.
target_fractions = [0.3, 0.5, 0.7]
target_means = [0.1, 0.2, 0.3]
target_maxs = [0.1, 0.5, 0.9]
fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharex=True)
for target_fraction in target_fractions:
    grad_level, U_z, info = find_gradient_level_for_sparsity_constraint(grad_U_matrix, delta_z_arr, U_arr, target_fraction=target_fraction)
    print(f"Sparsity constraint: target_fraction={target_fraction}, grad_level={grad_level:.4f}, final_fraction={info['final_fraction']:.4f}, converged={info['converged']}, iterations={info['iterations']}")
    axes[0].plot(delta_z_arr, U_z, label=f"sparsity: {target_fraction * 100:.0f}%")
for target_mean in target_means:
    grad_level, U_z, info = find_gradient_level_for_mean_constraint(grad_U_matrix, delta_z_arr, U_arr, target_mean=target_mean)
    print(f"Mean constraint: target_mean={target_mean}, grad_level={grad_level:.4f}, final_mean={info['mean']:.4f}, converged={info['converged']}, iterations={info['iterations']}")
    axes[1].plot(delta_z_arr, U_z, label=rf"$\langle U(\Delta z) \rangle = {target_mean}$")
for target_max in target_maxs:
    grad_level, U_z, info = find_gradient_level_for_maximum_constraint(grad_U_matrix, delta_z_arr, U_arr, target_max=target_max)
    print(f"Maximum constraint: target_max={target_max}, grad_level={grad_level:.4f}, final_max={info['mean']:.4f}, converged={info['converged']}, iterations={info['iterations']}")
    axes[2].plot(delta_z_arr, U_z, label=fr"$\max ~U(\Delta z) = {target_max}$")

# axes[0].set_title("U(z) under Sparsity Constraints")
axes[0].axvline(0, color='black', linestyle=':', linewidth=0.8)
axes[0].axhline(0, color='black', linestyle=':', linewidth=0.8)
axes[0].set_title('A')
axes[0].set_ylabel(r"$U(\Delta z)$")
axes[0].set_xlabel(r'Anti-causal $\leftarrow$ $\Delta z$ (rad) $\rightarrow$ Causal')
axes[0].legend()
# axes[1].set_title("U(z) under Mean Constraints")
axes[1].axvline(0, color='black', linestyle=':', linewidth=0.8)
axes[1].axhline(0, color='black', linestyle=':', linewidth=0.8)
axes[1].set_title('B')
axes[1].set_ylabel(r"$U(\Delta z)$")
axes[1].set_xlabel(r'Anti-causal $\leftarrow$ $\Delta z$ (rad) $\rightarrow$ Causal')
axes[1].legend()
# axes[2].set_title("U(z) under Maximum Constraints")
axes[2].axvline(0, color='black', linestyle=':', linewidth=0.8)
axes[2].axhline(0, color='black', linestyle=':', linewidth=0.8)
axes[2].set_title('C')
axes[2].set_xlabel(r'Anti-causal $\leftarrow$ $\Delta z$ (rad) $\rightarrow$ Causal')
axes[2].set_ylabel(r"$U(\Delta z)$")
axes[2].legend()
plt.tight_layout()
plt.savefig(FIGURE_DIR / 'suppl-optimal-U-various-constraints.pdf', bbox_inches='tight')
plt.show()


### Supplementary Figure: Optimal $U(\Delta z)$ under different input amplitudes

Reproduces Fig. S10 by changing the amplitude of the traveling-wave input and comparing the resulting temporal asymmetry.


In [ ]:
# Compare optimized U under different input amplitudes.
from copy import deepcopy
amp_values = [1.0, 2.0, 3.0]
U_z_arr = []
for amp in amp_values:
    params_mod = deepcopy(params)
    params_mod['amp'] = amp
    h_rect = lambda z, t: params_mod['amp'] * rectified_cos(2 * np.pi * params_mod['freq'] * t - z, theta_c=np.pi/2)
    h_prime = lambda z, t: np.where(h_rect(z, t) > 0, -params_mod['amp'], 0)

    U_arr = np.linspace(1e-3, 1.0-1e-3, 50)
    grad_U_matrix = compute_grad_U_matrix(h_rect, h_prime, delta_z_arr, U_arr, params_mod, w0_init= params_mod['w0_init'])
    grad_level, U_z, info = find_gradient_level_for_mean_constraint(grad_U_matrix, delta_z_arr, U_arr, target_mean=params_mod['U_init'])
    U_z_arr.append(U_z)

fig, ax = plt.subplots(figsize=(8, 6))
for i, amp in enumerate(amp_values):
    ax.plot(delta_z_arr, U_z_arr[i], label=f"A={amp}")
ax.axvline(0, color='black', linestyle=':', linewidth=0.8)
ax.axhline(0, color='black', linestyle=':', linewidth=0.8)
ax.set_xlabel(r'Anti-causal $\leftarrow$ $\Delta z$ (rad) $\rightarrow$ Causal')
ax.set_ylabel(r"$U(\Delta z)$")
# ax.set_title("U(z) under Different Input Amplitudes")
ax.legend(loc='upper left', bbox_to_anchor=(1.02, 1), borderaxespad=0)
plt.tight_layout()
plt.savefig(FIGURE_DIR / 'suppl-optimal-U-different-amp.pdf', bbox_inches='tight')
plt.show()
